In [1]:
import pandas as pd

In [2]:
import numpy as np

 1. Load & Profile

In [48]:
df = pd.read_csv("sales_messy.csv")

In [4]:
customers = pd.read_csv("customers.csv")

In [5]:
print(df.shape)

(208, 9)


In [6]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB
None


In [7]:
print(df.isnull().sum())

order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64


In [8]:
print(df['country'].unique())

<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str


2. Clean the Data

In [9]:
print(f"Total Duplicate Rows: {df.duplicated().sum()}")

Total Duplicate Rows: 8


In [10]:
df = df.drop_duplicates()

In [11]:
df['country'] = df['country'].astype(str).str.strip().str.title()

In [12]:
df['discount'] = df['discount'].fillna(0)

In [13]:
df['unit_price'] = df['unit_price'].fillna(df['unit_price'].median())

In [14]:
df = df.dropna(subset=['customer_id'])

In [15]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [16]:
print("Missing values after cleaning:")

Missing values after cleaning:


In [17]:
print(df[['discount', 'unit_price', 'customer_id']].isnull().sum())

discount       0
unit_price     0
customer_id    0
dtype: int64


3. Enrich

In [18]:
df['revenue'] = (df['quantity'] * df['unit_price']) * (1 - df['discount'])

In [19]:
df['month'] = df['order_date'].dt.strftime('%Y-%m')

In [20]:
df[['quantity', 'unit_price', 'discount', 'revenue', 'month']].head()

,quantity,unit_price,discount,revenue,month
1,3,59.99,0.10,161.973,2025-07
2,1,799.00,0.05,759.050,2025-02
3,4,899.00,0.20,2876.800,2025-12
4,5,549.00,0.10,2470.500,2025-08
5,6,59.99,0.15,305.949,2025-02


4. Merge

In [21]:
df['customer_id'] = df['customer_id'].astype(int)

In [22]:
customers['customer_id'] = customers['customer_id'].astype(int)

In [23]:
before_merge_count = df.shape[0]

In [24]:
df = df.merge(customers, on="customer_id", how="left")

In [25]:
assert df.shape[0] == before_merge_count, "Error: Row count changed! Check customers table for duplicate IDs."

In [26]:
print(f"Merge successful. Row count remained exactly {df.shape[0]}.")

Merge successful. Row count remained exactly 193.


 5. Aggregate

In [27]:
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()

In [28]:
print("--- REVENUE BY CATEGORY ---")

--- REVENUE BY CATEGORY ---


In [29]:
display(revenue_by_category)

,category,revenue
0,Laptops,161187.4000
1,Phones,63403.4000
2,Monitors,58295.5500
3,Accessories,10323.5465


In [30]:
revenue_by_month = df.groupby('month')['revenue'].sum().sort_values(ascending=False).reset_index()

In [31]:
print("\n--- REVENUE BY MONTH ---")


--- REVENUE BY MONTH ---


In [32]:
display(revenue_by_month)

,month,revenue
0,2025-07,42529.4260
1,2025-10,33697.7500
2,2025-08,30827.4315
3,2025-04,26456.2405
4,2025-06,24754.8375
5,2025-05,23633.5065
6,2025-12,22739.7510
7,2025-03,19836.5880
8,2025-02,19631.0780
9,2025-11,19117.3905


In [33]:
revenue_by_segment = df.groupby('segment')['revenue'].sum().sort_values(ascending=False).reset_index()

In [34]:
print("\n--- REVENUE BY SEGMENT ---")


--- REVENUE BY SEGMENT ---


In [35]:
display(revenue_by_segment)

,segment,revenue
0,Consumer,174850.8365
1,Education,77782.0320
2,Business,40577.0280


In [36]:
total_revenue = df['revenue'].sum()

In [37]:
top_category_name = revenue_by_category.iloc[0]['category']

In [38]:
top_category_rev = revenue_by_category.iloc[0]['revenue']

In [39]:
top_category_share = (top_category_rev / total_revenue) * 100

In [40]:
best_month_name = revenue_by_month.iloc[0]['month']

In [41]:
best_month_rev = revenue_by_month.iloc[0]['revenue']

In [42]:
top_segment_name = revenue_by_segment.iloc[0]['segment']

In [43]:
top_segment_rev = revenue_by_segment.iloc[0]['revenue']

In [44]:
print(f"Total Revenue: {total_revenue:,.2f}")

Total Revenue: 293,209.90


In [45]:
print(f"Top Category: {top_category_name} at {top_category_rev:,.2f} ({top_category_share:.1f}% of total)")

Top Category: Laptops at 161,187.40 (55.0% of total)


In [46]:
print(f"Best Month: {best_month_name} at {best_month_rev:,.2f}")

Best Month: 2025-07 at 42,529.43


In [47]:
print(f"Top Segment: {top_segment_name} at {top_segment_rev:,.2f}")

Top Segment: Consumer at 174,850.84
